# Phase 2: Temporal Feature Expansion

## Goal

Improve 60-minute-ahead household power forecasting by adding features that describe
recent changes, trends, volatility, and local extremes in power consumption.

The first modeling phase showed that:

- Histogram Gradient Boosting was the strongest validation model.
- The baseline PyTorch MLP performed close to HGB but did not outperform it.
- The neural network tended to underpredict large power-consumption events.
- Changing the neural-network loss function and dropout did not improve performance.

Instead of continuing to tune the model architecture, Phase 2 will test whether
better temporal information improves forecasting performance.

## Hypothesis

The existing lag and rolling-average features describe past power levels, but they
do not explicitly describe whether consumption is currently rising, falling, or
becoming unusually volatile.

Features representing recent changes, ranges, maxima, and variability may help the
models identify developing high-consumption periods.

## Experimental Plan

1. Load the validated minute-level dataset.
2. Recreate the existing lag, rolling, calendar, and target features.
3. Add new temporal features describing:
   - recent power changes
   - recent volatility
   - recent maximum power
   - recent minimum power
   - recent power range
4. Rebuild the modeling dataset without temporal leakage.
5. Recreate the same chronological train/validation/test split.
6. Retrain Histogram Gradient Boosting using the expanded feature set.
7. Retrain the baseline MLP architecture using the expanded feature set.
8. Compare validation performance against the original Phase 1 results.
9. Keep the test set untouched until Phase 2 model selection is complete.

## Phase 1 Benchmarks

- Histogram Gradient Boosting RMSE: 0.7885
- Baseline Neural Network RMSE: approximately 0.7983

The primary question is whether the expanded temporal features improve validation
RMSE beyond these benchmarks.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

data_path = Path(
    "../data/processed/household_power_validated.parquet"
)

df = pd.read_parquet(data_path)

df = (
    df.sort_values("datetime")
      .reset_index(drop=True)
)

print(df.shape)
print(df["datetime"].min())
print(df["datetime"].max())

(2075259, 10)
2006-12-16 17:24:00
2010-11-26 21:02:00


In [2]:
df["hour"] = df["datetime"].dt.hour
df["day_of_week"] = df["datetime"].dt.dayofweek
df["month"] = df["datetime"].dt.month

df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)

In [3]:
lag_minutes = [
    1,
    5,
    15,
    30,
    60,
    120,
    1380
]

for lag in lag_minutes:
    df[f"power_lag_{lag}"] = (
        df["Global_active_power"].shift(lag)
    )

In [4]:
df["rolling_mean_15"] = (
    df["Global_active_power"]
    .rolling(15)
    .mean()
)

df["rolling_mean_60"] = (
    df["Global_active_power"]
    .rolling(60)
    .mean()
)

df["rolling_std_60"] = (
    df["Global_active_power"]
    .rolling(60)
    .std()
)

df["rolling_mean_1440"] = (
    df["Global_active_power"]
    .rolling(1440)
    .mean()
)

In [5]:
df["power_target_60"] = (
    df["Global_active_power"].shift(-60)
)

In [7]:
# change features

# power_change_1  > 0 → power increased during the last minute
# power_change_15 > 0 → power increased over the last 15 minutes
# power_change_60 < 0 → power decreased over the last hour

df["power_change_1"] = (
    df["Global_active_power"]
    - df["power_lag_1"]
)

df["power_change_5"] = (
    df["Global_active_power"]
    - df["power_lag_5"]
)

df["power_change_15"] = (
    df["Global_active_power"]
    - df["power_lag_15"]
)

df["power_change_60"] = (
    df["Global_active_power"]
    - df["power_lag_60"]
)

In [8]:
# volatility and extreme-value features

df["rolling_std_15"] = (
    df["Global_active_power"]
    .rolling(15)
    .std()
)

df["rolling_max_15"] = (
    df["Global_active_power"]
    .rolling(15)
    .max()
)

df["rolling_max_60"] = (
    df["Global_active_power"]
    .rolling(60)
    .max()
)

df["rolling_min_60"] = (
    df["Global_active_power"]
    .rolling(60)
    .min()
)

In [9]:
# range feature

df["rolling_range_60"] = (
    df["rolling_max_60"]
    - df["rolling_min_60"]
)

In [10]:
# rolling_std_15
#     → how unstable power has been recently

# rolling_max_15
#     → recent short-term peak

# rolling_max_60
#     → highest power observed during the past hour

# rolling_min_60
#     → lowest power observed during the past hour

# rolling_range_60
#     → how widely consumption has moved during the past hour